# 实验总要求
## 通用实验开发要求

1. 实验入口
- 只保留 `pytest` 作为唯一运行入口。
- 不提供脚本直跑入口，不写 `main()` / `if __name__ == "__main__"`。
- 不使用环境变量控制是否跳过实验；默认依赖服务已经启动且可用。
- 如果依赖服务不可用，测试应直接失败，暴露真实集成问题。

2. 测试编号
- 每个实验函数必须编号，从 `test_01_xxx` 开始递增。
- 抓包文件和汇总 key 也使用相同编号，例如：
- `01_xxx_capture.json`
- `test_01_xxx`
- 不要复用含义已经变化的旧编号名称；如果实验语义变了，同步改函数名、summary key、文档标题。

3. 注释规范
- 每个测试函数必须用中文 docstring 写清：
- 测试目的
- 实验方法
- 实验结果
- 关键 helper 函数也要写中文 docstring，说明：
- 目的
- 手段
- 返回结果
- 工具函数附近用简短中文注释说明：
- 这个工具用于哪个实验
- 模拟什么场景
- 预期输出是什么

4. pytest 返回值
- pytest 测试函数必须返回 `None`。
- 如果需要保存实验结果，使用模块级全局变量，例如：
    EXPERIMENT_RESULTS: dict[str, Any] = {}
- 每个测试结束时调用统一函数写入汇总文件：
    record_experiment_result("test_01_xxx", result)
- 每次写入都覆盖保存完整 JSON，保证中途失败时已完成实验仍有结果。

5. 断言要求

- 实验不能只停留在文档说明，必须实际运行并断言关键行为。
- 对比实验要断言核心字段一致或差异符合预期。
- 对“模型可能波动”的部分，只断言结构性、协议性、确定性行为。

6. 文档要求

- 维护一份中文 markdown 最佳实践文档。
- 文档中每个实验都要写：
    - 实验函数名
    - 目的
    - 方法
    - 实测结果
    - 抓包路径
    - 工程结论
- 文档结尾维护“已验证结论清单”。
- 代码逻辑变化后，文档必须同步更新，不能留下旧实验描述。

7. 验证命令
每次修改后至少运行：
    python -m py_compile <实验脚本> <相关辅助脚本>
    python -m ruff check <实验脚本> <相关辅助脚本>
    python -m black --check <实验脚本> <相关辅助脚本>
    pytest <实验脚本> -q
如果 black --check 失败，先运行：
    python -m black <实验脚本> <相关辅助脚本>
然后重新跑完整验证.

# 回合1
## 参考资料：
zhuxs_learn/03_tool/langchain_tool_call.md 是vllm官方提供的工具调用文档，介绍了vllm支持的工具调用方式，包括命名函数调用、自动选择、必选和无工具选项等。
zhuxs_learn/03_tool/langchain_tool_call.md 是langchain官方提供的工具调用文档，包括langchain实现工具调用的多种方式，工具访问运行时上下文状态信息，工具调用错误处理，工具并行执行，以及这些的最佳实践ToolNode（可以细粒度的定义工具调用和前后处理各种逻辑）
/data/zhuxs/llm_agent/langchain_docs/build/oss/python 是langchain的官方文档，所有线上文档都可以在这里找到，建议先查看这里的mdx文档获取langchain推荐的实践方式
环境已经安装了langchain和langgraph，可以直接查看源码
/data/zhuxs/llm_agent/langchain_docs/build/oss/python/langchain/mcp.mdx 是langchain官方提供的mcp文档，介绍了mcp的使用方式和最佳实践，可以参考这个文档来实现通过mcp调用具的方式
zhuxs_learn/02_structure_output/final_structure_output.py 定义了通过langchain和openai sdk访问本地部署的vllm模型的方式（支持工具调用）并设置http抓包保存方便审核对比，以此为基准开展实验

## 任务目标：探究langchain+vllm工具调用的最佳实践
1. vllm支持的工具并行调用（一次调用多个工具），自动选择，必选分别使用openai sdk和langchain实现，通过抓包确保二者调用方式一致
2. langchain官方给出的工具访问运行时上下文状态信息，工具调用错误处理，工具并行执行，以及这些的最佳实践ToolNode给出vllm后端时的示例并实际跑一遍抓取结果，确保这些功能正常而不只是停留在文档层面（你可以设置一些模拟场景进行测试，比如工具调用的时长判断是否并行执行，比如故意让工具调用出错查看如何处理）
3. 除了传统的工具调用，langchain还支持通过mcp来调用工具，在mcp上实现工具调用的方式并实际跑一遍抓取结果，确保这种方式也能正常使用，并且和传统工具调用方式的结果一致（测试上下文状态信息，工具调用错误处理，工具并行执行，以及这些的最佳实践ToolNode都兼容mcp），另外我理解mcp最大的优势就是可以异步调用，因此mcp一定要实现异步

## 任务要求：
启动subagent进行并行探索，并将结果汇总整理成清晰的中文md文档，每一个测试函数和对比实验都要有清晰的中文注释说明实验目的和实验方法细节和实验结果，最后形成一个完整的工具调用最佳实践指南，供后续参考和使用。






# 回合2
claude sonnet(使用claude sdk)
最高要求：通过AskUserQuestion工具与我交互直到我在其中选择`本次会话结束`选项你才能结束，你的任何问题和反馈都必须调用AskUserQuestion工具来呈现，绝对不可以擅自中断或者进行反问！！！
zhuxs_learn/03_tool 里面是langchain+vllm+mcp的tool calling的功能实验和测试，用于教学，但是现在代码的注释写的不是很清晰，现在需要你丰富中文注释，给每一个函数和测试都加上注释方便一眼看出这个函数的目的，手段和结果







# 回合3
模型能否看到tools里面的description
    设计一个单独的实验证明，就用langchain+auto+vllm


required/none/named是类似约束解码一样必须遵守还是者说模型会根据description来判断是否调用工具，根据zhuxs_learn/03_tool/vllm_tool_calling.md和vllm仓库的描述
    - named、required、none 在 vLLM 里都不是“建议”，而是明确模式。
    - 但 required 的“保证”是 API/解码层面的强约束，不是“语义上合理才会调工具”。
    - 所以：如果你给了一个本来不该调用函数的 prompt，但 tool_choice="required"，正常完成时，vLLM 会强制返回一个或多个 tool call，而不是自由文本。
    - named：保证返回“可解析”的函数调用，但“不保证质量高” (zhuxs_learn/03_tool/vllm_tool_calling.md:88-94)
    - required：保证生成一个或多个 tool call (zhuxs_learn/03_tool/vllm_tool_calling.md:99-104)
    - none：保证不生成 tool call，只返回普通文本 (zhuxs_learn/03_tool/vllm_tool_calling.md:105-110)
    - 这三者里，只有 auto 不是强约束；auto 下参数甚至可能不合法 (zhuxs_learn/03_tool/vllm_tool_calling.md:112-123)
    要补一个工程上的例外：
    - 这不是数学意义上的 100% 无例外。
    - 如果输出被 max_tokens / max_completion_tokens / max_output_tokens 截断，官方测试里就有 tool_choice="required" 但最终没有 function_call、而是 incomplete_details.reason == "max_output_tokens" 的情况 (vllm-project/vllm/tests/entrypoints/openai/responses/test_function_call.py:157)
    - 也就是说：required 是“正常完成时强制 tool call”，不是“任何异常情况下都一定有 tool call”。
    一句话总结：
    - required：强制调用，哪怕语义上不该调，也会尽量凑出一个调用
    - named：强制调你指定的那个函数
    - none：强制不调
    - 真正“让模型自己决定调不调”的只有 auto

    这里同样需要设计实验证明（因为我们已经证明了langchain和openai sdk的一致性，所以只需要给named、required、none分别加一个langchain的实验证明，不需要openai再重复一遍）


runtime: ToolRuntime最大的作用是访问状态。此参数会自动注入并隐藏于 LLM - 它不会出现在工具的架构中，但是目前设计的traditional实验根本没有涉及llm的使用和抓包来验证该参数对llm不可见，目前 zhuxs_learn/03_tool/langchain_tool_call.md 给出的都是封装好的create_agent没有内部细节，我希望你调研create_agent其中原理给我一个最简但是完整的langgraph图实现，包括输入，llm调用工具（toolruntime不可见），工具执行并返回，llm。。。输出的完整流程图。我希望工具执行这里你给我两个实现一个ToolNode一个自定义工具执行node，因为我目前看ToolNode依然封装度过高，比如我可能对多个tool的执行顺序写逻辑做判断，这里ToolNode就不太能做到（？这一点存疑，需要你调查告诉我），自定义工具执行node则完全不封装，完全由我来控制工具的执行细节和前后逻辑（包括并行执行），以及Command更新之类的。但是我希望你的custom node能兼容ToolNode的输入输出格式（你需要具体调研一下ToolNode），这样我就可以在同一个图里同时使用ToolNode和custom node来实现不同程度的封装和控制。